# Section 9: Augmentation Strategy
Defines and validates the training-only augmentation pipeline. Augmentation is applied after Section 8 preprocessing and before model input. Val and test splits are never augmented.


In [1]:
%run 01_config.ipynb

import os
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from IPython.display import display


Creating output structure in: D:\GradProj\Skin Cancer Dataset\pipeline_output
Set basic random seeds to 42.
No GPU detected. Processing on CPU.
Mixed precision (mixed_float16) enabled natively.


## Check for TensorFlow Addons (rotation support)


In [2]:
USE_TFA_ROTATE = False
try:
    import tensorflow_addons as tfa
    USE_TFA_ROTATE = True
    print("TensorFlow Addons available. Using tfa.image.rotate for fine-grained rotation.")
except ImportError:
    print("TensorFlow Addons not available. Falling back to tf.image.rot90 (90-degree increments).")
    print("For fine-grained rotation, install: pip install tensorflow-addons")


TensorFlow Addons not available. Falling back to tf.image.rot90 (90-degree increments).
For fine-grained rotation, install: pip install tensorflow-addons


## Load Split Manifests & Validate Counts


In [3]:
d_splits = os.path.join(OUTPUT_ROOT, "splits")
d_augment = os.path.join(OUTPUT_ROOT, "audit_reports", "augmentation")
os.makedirs(d_augment, exist_ok=True)

df_train = pd.read_csv(os.path.join(d_splits, "train_manifest.csv"))
df_val = pd.read_csv(os.path.join(d_splits, "val_manifest.csv"))
df_test = pd.read_csv(os.path.join(d_splits, "test_manifest.csv"))

expected = {"train": 14416, "val": 3000, "test": 3097}
splits = {"train": df_train, "val": df_val, "test": df_test}
errors = []

print("=== SECTION 9 INPUT DATASET VALIDATION ===")
for name, df in splits.items():
    actual = len(df)
    print(f"{name}: {actual} rows (Expected: {expected[name]})")
    if actual != expected[name]:
        errors.append(f"{name} row mismatch: expected {expected[name]}, got {actual}")

if errors:
    raise ValueError(
        "Section 9 must run on the frozen split manifests, but mismatches were found:\n- "
        + "\n- ".join(errors)
    )

print("\nAll split manifest counts match expected frozen state.")


=== SECTION 9 INPUT DATASET VALIDATION ===
train: 14416 rows (Expected: 14416)
val: 3000 rows (Expected: 3000)
test: 3097 rows (Expected: 3097)

All split manifest counts match expected frozen state.


## Import Section 8 Preprocessing Function


In [4]:
# Re-define Section 8 preprocessing function inline for self-contained execution.
# In production pipeline (Section 10), this will be imported or %run from 08_preprocessing.

PREPROCESS_TARGET_SIZE = IMAGE_SIZE[0]  # 224
PREPROCESS_INTERPOLATION = "bilinear"

def preprocess_image(image_path, target_size=PREPROCESS_TARGET_SIZE, preprocess_fn=None):
    """Load, resize-and-center-crop, and normalize a single image (Section 8 contract)."""
    raw_bytes = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(raw_bytes, channels=3)

    shape = tf.shape(image)
    h = tf.cast(shape[0], tf.float32)
    w = tf.cast(shape[1], tf.float32)
    target_f = tf.cast(target_size, tf.float32)

    scale = target_f / tf.minimum(h, w)
    new_h = tf.cast(tf.math.ceil(h * scale), tf.int32)
    new_w = tf.cast(tf.math.ceil(w * scale), tf.int32)

    image = tf.image.resize(image, [new_h, new_w], method=PREPROCESS_INTERPOLATION)
    image = tf.image.resize_with_crop_or_pad(image, target_size, target_size)
    image = tf.cast(image, tf.float32)

    if preprocess_fn is not None:
        image = preprocess_fn(image)
    else:
        image = image / 255.0

    return image

def preprocess_image_default(image_path):
    return preprocess_image(image_path, target_size=PREPROCESS_TARGET_SIZE, preprocess_fn=None)

print("Section 8 preprocessing function loaded.")


Section 8 preprocessing function loaded.


## Define Augmentation Policy Constants


In [5]:
# Augmentation policy — all decisions documented in section_9_developer_ready_definitions.md
AUG_HORIZONTAL_FLIP = True
AUG_VERTICAL_FLIP = True
AUG_ROTATION_MAX_DEGREES = 30.0
AUG_BRIGHTNESS_RANGE = (0.85, 1.15)
AUG_CONTRAST_RANGE = (0.85, 1.15)
AUG_SATURATION_RANGE = (0.85, 1.15)
AUG_HUE_MAX_DELTA = 0.02
AUG_CLIP_MIN = 0.0
AUG_CLIP_MAX = 1.0

print("=== AUGMENTATION POLICY ===")
print(f"Horizontal flip: {AUG_HORIZONTAL_FLIP}")
print(f"Vertical flip: {AUG_VERTICAL_FLIP}")
print(f"Rotation: ±{AUG_ROTATION_MAX_DEGREES}°")
print(f"Brightness factor: {AUG_BRIGHTNESS_RANGE}")
print(f"Contrast factor: {AUG_CONTRAST_RANGE}")
print(f"Saturation factor: {AUG_SATURATION_RANGE}")
print(f"Hue max delta: {AUG_HUE_MAX_DELTA}")
print(f"Final clip range: [{AUG_CLIP_MIN}, {AUG_CLIP_MAX}]")


=== AUGMENTATION POLICY ===
Horizontal flip: True
Vertical flip: True
Rotation: ±30.0°
Brightness factor: (0.85, 1.15)
Contrast factor: (0.85, 1.15)
Saturation factor: (0.85, 1.15)
Hue max delta: 0.02
Final clip range: [0.0, 1.0]


## Define Augmentation Function


In [6]:
def augment_training_image(image, clip_min=AUG_CLIP_MIN, clip_max=AUG_CLIP_MAX):
    """
    Apply stochastic augmentation to a single preprocessed training image.

    Args:
        image: Tensor of shape (224, 224, 3), dtype float32,
               already preprocessed by Section 8.
        clip_min: float, lower bound for final clipping (default 0.0 for [0,1] normalization).
        clip_max: float, upper bound for final clipping (default 1.0 for [0,1] normalization).

    Returns:
        Tensor of shape (224, 224, 3), dtype float32, augmented and clipped.
    """
    # 1. Random horizontal flip
    image = tf.image.random_flip_left_right(image)

    # 2. Random vertical flip
    image = tf.image.random_flip_up_down(image)

    # 3. Random rotation
    if USE_TFA_ROTATE:
        max_radians = AUG_ROTATION_MAX_DEGREES * (np.pi / 180.0)
        angle = tf.random.uniform([], -max_radians, max_radians)
        image = tfa.image.rotate(image, angle, interpolation="bilinear", fill_mode="reflect")
    else:
        # Fallback: random 0/90/180/270 degree rotation
        k = tf.random.uniform([], 0, 4, dtype=tf.int32)
        image = tf.image.rot90(image, k=k)

    # 4. Random brightness
    brightness_factor = tf.random.uniform([], AUG_BRIGHTNESS_RANGE[0], AUG_BRIGHTNESS_RANGE[1])
    image = image * brightness_factor

    # 5. Random contrast
    image = tf.image.random_contrast(image, AUG_CONTRAST_RANGE[0], AUG_CONTRAST_RANGE[1])

    # 6. Random saturation
    image = tf.image.random_saturation(image, AUG_SATURATION_RANGE[0], AUG_SATURATION_RANGE[1])

    # 7. Random hue
    image = tf.image.random_hue(image, AUG_HUE_MAX_DELTA)

    # 8. Clip to valid range
    image = tf.clip_by_value(image, clip_min, clip_max)

    return image


print("Augmentation function defined.")
if USE_TFA_ROTATE:
    print("Rotation mode: tfa.image.rotate (fine-grained, reflect fill)")
else:
    print("Rotation mode: tf.image.rot90 (90-degree increments, fallback)")


Augmentation function defined.
Rotation mode: tf.image.rot90 (90-degree increments, fallback)


## Validation Check 1: Shape, Dtype, and Value Range Preservation


In [7]:
validation_results = []

# Preprocess a sample training image, then augment it
sample_path = df_train.iloc[0]["full_path"]
preprocessed = preprocess_image_default(sample_path)

# Run augmentation 10 times to check consistency
print("Running augmentation on a single image 10 times...")
all_shapes_ok = True
all_dtypes_ok = True
all_ranges_ok = True

for i in range(10):
    aug = augment_training_image(preprocessed)
    s_ok = (aug.shape == (PREPROCESS_TARGET_SIZE, PREPROCESS_TARGET_SIZE, 3))
    d_ok = (aug.dtype == tf.float32)
    min_v = float(tf.reduce_min(aug).numpy())
    max_v = float(tf.reduce_max(aug).numpy())
    r_ok = (min_v >= AUG_CLIP_MIN and max_v <= AUG_CLIP_MAX)
    if not s_ok: all_shapes_ok = False
    if not d_ok: all_dtypes_ok = False
    if not r_ok: all_ranges_ok = False

print(f"Shape preserved across 10 augmentations: {all_shapes_ok}")
print(f"Dtype preserved across 10 augmentations: {all_dtypes_ok}")
print(f"Value range [{AUG_CLIP_MIN}, {AUG_CLIP_MAX}] preserved: {all_ranges_ok}")

validation_results.append({"check": "shape_preservation", "expected": f"({PREPROCESS_TARGET_SIZE},{PREPROCESS_TARGET_SIZE},3)", "actual": "consistent" if all_shapes_ok else "inconsistent", "pass": all_shapes_ok})
validation_results.append({"check": "dtype_preservation", "expected": "float32", "actual": "consistent" if all_dtypes_ok else "inconsistent", "pass": all_dtypes_ok})
validation_results.append({"check": "value_range_clipping", "expected": f"[{AUG_CLIP_MIN}, {AUG_CLIP_MAX}]", "actual": "within bounds" if all_ranges_ok else "out of bounds", "pass": all_ranges_ok})


Running augmentation on a single image 10 times...
Shape preserved across 10 augmentations: True
Dtype preserved across 10 augmentations: True
Value range [0.0, 1.0] preserved: True


## Validation Check 2: Visual Diversity Grid (Same Image, 9 Augmentations)


In [8]:
def generate_diversity_grid(image_path, title, save_path, n=9):
    """Show the same image augmented n times to demonstrate stochastic diversity."""
    preprocessed = preprocess_image_default(image_path)

    cols = 3
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(10, 10))
    fig.suptitle(title, fontsize=16)

    for i in range(n):
        ax = axes[i // cols][i % cols]
        aug = augment_training_image(preprocessed)
        ax.imshow(aug.numpy())
        ax.set_title(f"Aug {i+1}", fontsize=10)
        ax.axis("off")

    # Turn off unused axes
    for i in range(n, rows * cols):
        axes[i // cols][i % cols].axis("off")

    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"Saved: {save_path}")


# Use a training image that should show clear visual differences under augmentation
diversity_path = df_train.iloc[0]["full_path"]
generate_diversity_grid(
    diversity_path,
    "Augmentation Diversity: Same Image × 9",
    os.path.join(d_augment, "augmentation_diversity_grid.png")
)

# Check that augmentations actually produce different outputs
preprocessed = preprocess_image_default(diversity_path)
aug_a = augment_training_image(preprocessed)
aug_b = augment_training_image(preprocessed)
diversity_ok = not bool(tf.reduce_all(tf.equal(aug_a, aug_b)).numpy())
print(f"Two augmentations of the same image differ: {diversity_ok}")

validation_results.append({"check": "augmentation_diversity", "expected": "different outputs", "actual": "different" if diversity_ok else "identical", "pass": diversity_ok})


Saved: D:\GradProj\Skin Cancer Dataset\pipeline_output\audit_reports\augmentation\augmentation_diversity_grid.png
Two augmentations of the same image differ: True


## Validation Check 3: Per-Class Augmentation Grid


In [9]:
def generate_per_class_aug_grid(df, title, save_path, n_aug=3):
    """Show one image per class, each augmented n_aug times."""
    fig, axes = plt.subplots(len(CLASS_NAMES), n_aug, figsize=(10, 10))
    fig.suptitle(title, fontsize=16)

    for row_idx, cls in enumerate(CLASS_NAMES):
        cls_df = df[df["final_authoritative_label"] == cls]
        sample_row = cls_df.sample(1, random_state=RANDOM_SEED).iloc[0]
        preprocessed = preprocess_image_default(sample_row["full_path"])

        for col_idx in range(n_aug):
            ax = axes[row_idx][col_idx]
            aug = augment_training_image(preprocessed)
            ax.imshow(aug.numpy())
            if col_idx == 0:
                ax.set_ylabel(cls, fontsize=14, rotation=0, labelpad=40)
            ax.set_title(f"Aug {col_idx+1}", fontsize=10)
            ax.axis("off")

    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"Saved: {save_path}")


generate_per_class_aug_grid(
    df_train,
    "Augmented Samples by Class (Training)",
    os.path.join(d_augment, "augmentation_per_class_grid.png")
)

validation_results.append({"check": "per_class_grid_generated", "expected": "True", "actual": "True", "pass": True})


Saved: D:\GradProj\Skin Cancer Dataset\pipeline_output\audit_reports\augmentation\augmentation_per_class_grid.png


## Validation Check 4: Statistical Sanity (Batch-Level Mean/Std)


In [10]:
# Preprocess and augment a batch of 100 training images, check pixel statistics are plausible
batch_size_check = 100
sample_df = df_train.sample(batch_size_check, random_state=RANDOM_SEED)

aug_means = []
aug_stds = []

for _, row in sample_df.iterrows():
    preprocessed = preprocess_image_default(row["full_path"])
    aug = augment_training_image(preprocessed)
    aug_means.append(float(tf.reduce_mean(aug).numpy()))
    aug_stds.append(float(tf.math.reduce_std(aug).numpy()))

batch_mean = np.mean(aug_means)
batch_std = np.mean(aug_stds)

# For [0,1] normalized dermoscopic images, expect:
#   mean roughly 0.4-0.7 (skin tones dominate)
#   std roughly 0.05-0.3
mean_plausible = (0.1 < batch_mean < 0.9)
std_plausible = (0.01 < batch_std < 0.5)
stats_ok = mean_plausible and std_plausible

print(f"=== STATISTICAL SANITY CHECK (n={batch_size_check}) ===")
print(f"Mean of per-image means: {batch_mean:.4f} (plausible: {mean_plausible})")
print(f"Mean of per-image stds:  {batch_std:.4f} (plausible: {std_plausible})")
print(f"Overall plausible: {stats_ok}")

validation_results.append({"check": "statistical_sanity_mean", "expected": "0.1-0.9", "actual": f"{batch_mean:.4f}", "pass": mean_plausible})
validation_results.append({"check": "statistical_sanity_std", "expected": "0.01-0.5", "actual": f"{batch_std:.4f}", "pass": std_plausible})


=== STATISTICAL SANITY CHECK (n=100) ===
Mean of per-image means: 0.5361 (plausible: True)
Mean of per-image stds:  0.1909 (plausible: True)
Overall plausible: True


## Save Augmentation Policy & Validation Report


In [11]:
# Save augmentation policy decisions as a CSV artifact
policy_data = [
    {"operation": "horizontal_flip", "enabled": True, "probability": 0.5, "parameters": "n/a"},
    {"operation": "vertical_flip", "enabled": True, "probability": 0.5, "parameters": "n/a"},
    {"operation": "rotation", "enabled": True, "probability": 1.0,
     "parameters": f"max_degrees={AUG_ROTATION_MAX_DEGREES}, mode={'tfa_reflect' if USE_TFA_ROTATE else 'rot90_fallback'}"},
    {"operation": "brightness", "enabled": True, "probability": 1.0,
     "parameters": f"factor_range={AUG_BRIGHTNESS_RANGE}"},
    {"operation": "contrast", "enabled": True, "probability": 1.0,
     "parameters": f"factor_range={AUG_CONTRAST_RANGE}"},
    {"operation": "saturation", "enabled": True, "probability": 1.0,
     "parameters": f"factor_range={AUG_SATURATION_RANGE}"},
    {"operation": "hue", "enabled": True, "probability": 1.0,
     "parameters": f"max_delta={AUG_HUE_MAX_DELTA}"},
    {"operation": "final_clip", "enabled": True, "probability": 1.0,
     "parameters": f"range=[{AUG_CLIP_MIN}, {AUG_CLIP_MAX}]"},
    {"operation": "zoom", "enabled": False, "probability": 0.0, "parameters": "excluded_v1"},
    {"operation": "translation", "enabled": False, "probability": 0.0, "parameters": "excluded_v1"},
    {"operation": "cutout", "enabled": False, "probability": 0.0, "parameters": "forbidden_medical"},
    {"operation": "mixup", "enabled": False, "probability": 0.0, "parameters": "forbidden_medical"},
    {"operation": "channel_shuffle", "enabled": False, "probability": 0.0, "parameters": "forbidden_medical"},
    {"operation": "elastic_deformation", "enabled": False, "probability": 0.0, "parameters": "forbidden_medical"},
    {"operation": "grayscale", "enabled": False, "probability": 0.0, "parameters": "forbidden_medical"},
]

df_policy = pd.DataFrame(policy_data)
policy_path = os.path.join(d_augment, "augmentation_policy.csv")
df_policy.to_csv(policy_path, index=False)
print(f"Augmentation policy saved to: {policy_path}")
display(df_policy)


Augmentation policy saved to: D:\GradProj\Skin Cancer Dataset\pipeline_output\audit_reports\augmentation\augmentation_policy.csv


,operation,enabled,probability,parameters
0,horizontal_flip,True,0.5,n/a
1,vertical_flip,True,0.5,n/a
2,rotation,True,1.0,"max_degrees=30.0, mode=rot90_fallback"
3,brightness,True,1.0,"factor_range=(0.85, 1.15)"
4,contrast,True,1.0,"factor_range=(0.85, 1.15)"
5,saturation,True,1.0,"factor_range=(0.85, 1.15)"
6,hue,True,1.0,max_delta=0.02
7,final_clip,True,1.0,"range=[0.0, 1.0]"
8,zoom,False,0.0,excluded_v1
9,translation,False,0.0,excluded_v1


In [12]:
# Save validation report
df_validation = pd.DataFrame(validation_results)
validation_path = os.path.join(d_augment, "augmentation_validation_report.csv")
df_validation.to_csv(validation_path, index=False)
print(f"Validation report saved to: {validation_path}")
display(df_validation)

all_passed = df_validation["pass"].all()
print(f"\nAll validation checks passed: {all_passed}")

if not all_passed:
    failed = df_validation[df_validation["pass"] == False]
    print("FAILED CHECKS:")
    display(failed)


Validation report saved to: D:\GradProj\Skin Cancer Dataset\pipeline_output\audit_reports\augmentation\augmentation_validation_report.csv


,check,expected,actual,pass
0,shape_preservation,"(224,224,3)",consistent,True
1,dtype_preservation,float32,consistent,True
2,value_range_clipping,"[0.0, 1.0]",within bounds,True
3,augmentation_diversity,different outputs,different,True
4,per_class_grid_generated,True,True,True
5,statistical_sanity_mean,0.1-0.9,0.5361,True
6,statistical_sanity_std,0.01-0.5,0.1909,True



All validation checks passed: True


## Section 9 Summary


In [14]:
print("=== SECTION 9 FINAL SUMMARY ===")
print(f"Augmentation applied to: training split only")
print(f"Val/test augmentation: disabled (frozen rule)")
print(f"")
print(f"Enabled operations:")
print(f"  Horizontal flip (p=0.5)")
print(f"  Vertical flip (p=0.5)")
if USE_TFA_ROTATE:
    print(f"  Rotation ±{AUG_ROTATION_MAX_DEGREES}° (tfa, reflect fill)")
else:
    print(f"  Rotation 90° increments (rot90 fallback)")
print(f"  Brightness factor {AUG_BRIGHTNESS_RANGE}")
print(f"  Contrast factor {AUG_CONTRAST_RANGE}")
print(f"  Saturation factor {AUG_SATURATION_RANGE}")
print(f"  Hue delta ±{AUG_HUE_MAX_DELTA}")
print(f"  Final clip [{AUG_CLIP_MIN}, {AUG_CLIP_MAX}]")
print(f"")
print(f"Forbidden operations: cutout, mixup, channel_shuffle, elastic_deform, grayscale")
print(f"Excluded from v1: zoom, translation")
print(f"")
print(f"All validation checks passed: {all_passed}")

print("\nSaved files:")
print("- augmentation_policy.csv")
print("- augmentation_validation_report.csv")
print("- augmentation_diversity_grid.png")
print("- augmentation_per_class_grid.png")

print("\n=== SECTION 9 DOWNSTREAM CONTRACT ===")
print("Section 10 must apply augment_training_image() ONLY to training batches.")
print("Val and test batches receive preprocessing only (Section 8), no augmentation.")
print("If the backbone's preprocess_fn changes the value range, update clip_min/clip_max accordingly.")

print("\nSection 9 completed. No raw images were modified on disk.")


=== SECTION 9 FINAL SUMMARY ===
Augmentation applied to: training split only
Val/test augmentation: disabled (frozen rule)

Enabled operations:
  Horizontal flip (p=0.5)
  Vertical flip (p=0.5)
  Rotation 90° increments (rot90 fallback)
  Brightness factor (0.85, 1.15)
  Contrast factor (0.85, 1.15)
  Saturation factor (0.85, 1.15)
  Hue delta ±0.02
  Final clip [0.0, 1.0]

Forbidden operations: cutout, mixup, channel_shuffle, elastic_deform, grayscale
Excluded from v1: zoom, translation

All validation checks passed: True

Saved files:
- augmentation_policy.csv
- augmentation_validation_report.csv
- augmentation_diversity_grid.png
- augmentation_per_class_grid.png

=== SECTION 9 DOWNSTREAM CONTRACT ===
Section 10 must apply augment_training_image() ONLY to training batches.
Val and test batches receive preprocessing only (Section 8), no augmentation.
If the backbone's preprocess_fn changes the value range, update clip_min/clip_max accordingly.

Section 9 completed. No raw images were 